# DeepHit Toxic-CIF Grid Search Analysis

This notebook is for the 2D strategy grid produced by `DeepHitToxicCIFDecisionLogic`: `max_toxic_cif` x `horizon_index`.

The goal is compact final-setting selection:

1. Load one ticker/model grid-search CSV.
2. Rank feasible configurations with deterministic tie-breaks.
3. Visualize objective values and participation over the 2D grid.
4. Loop through all ticker/model result files and export selected settings.


In [ ]:
from pathlib import Path
import os
import sys
import tempfile

os.environ.setdefault("MPLCONFIGDIR", str(Path(tempfile.gettempdir()) / "matplotlib"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

PROJECT_DIR = Path.cwd()
if PROJECT_DIR.parent.name == "notebooks":
    PROJECT_DIR = PROJECT_DIR.parent
    PROJECT_DIR = PROJECT_DIR.parent
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

sns.set_theme(style="whitegrid", context="notebook")
pd.set_option("display.max_columns", 140)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")


In [ ]:
# ---- User configuration ----
TICKER = "AAPL"
MODEL_TYPE = "gru_transformer"

# Expected tree: OUTPUT_ROOT / TICKER / MODEL_TYPE / deephit_toxic_cif_grid_search.csv
OUTPUT_ROOT = PROJECT_DIR / "reports" / "deephit_bt_grid_search_toxic_cif_csv"

# Optional explicit file override. Leave as None for default path discovery.
RESULT_PATH_OVERRIDE = None

DATA_ROOT = PROJECT_DIR / "data"
DATA_START_DATE = "2025-10-01"
DATA_END_DATE = "2026-01-01"
RUNTIME_NPZ_PATH_OVERRIDE = None

# Objective: lower is better. Common choices:
# "mean_is_bps", "median_is_bps", "time_weighted_mean_is_bps", "toxic_cost_mean_bps"
OBJECTIVE_METRIC = "toxic_cost_mean_bps"

# Feasibility filters. Set to None to disable.
# Real fill rate is submitted_rate * fill_rate, i.e. fills / all candidate orders.
MIN_REAL_FILL_RATE = 0
MIN_SUBMITTED_RATE = None
MIN_FILL_RATE = None
MAX_METRIC_FAILED_RATE = None
MIN_SUBMITTED = None

# Robust selection: choose among feasible rows within this many bps of the best
# objective, then use stability-oriented tie-breaks. Set to 0 for pure best row.
NEAR_BEST_TOL_BPS = 0

# Tie-break policy inside the near-best band: prefer fewer failures, then more
# realized fills, less aggressive filtering, a shorter horizon, and finally the
# lower objective if still tied.
TIEBREAK_ASCENDING = {
    "metric_failed_rate": True,
    "real_fill_rate": False,
    "max_toxic_cif": False,
    "horizon_index": True,
    "objective": True,
}

EXPORT_SELECTED_PATH = PROJECT_DIR / "reports" / "final_backtest" / "optimal_toxic_cif_strategy.csv"


In [ ]:
COLUMN_ALIASES = {
    "submitted_orders": "submitted",
    "skipped_orders": "skipped",
    "filled_orders": "filled",
    "unfilled_orders": "unfilled",
    "canceled_orders": "canceled",
    "metric_ok_orders": "metric_ok",
    "metric_failed_orders": "metric_failed",
    "mean_implementation_shortfall_bps": "mean_is_bps",
    "median_implementation_shortfall_bps": "median_is_bps",
    "total_implementation_shortfall": "total_is",
    "total_implementation_shortfall_raw": "total_is_raw",
}

REQUIRED_GRID_COLUMNS = ["max_toxic_cif", "horizon_index"]


def compact_date(date: str) -> str:
    return str(date).replace("-", "")


def default_result_path(ticker: str, model_type: str) -> Path:
    return OUTPUT_ROOT / ticker.upper() / model_type / "deephit_toxic_cif_grid_search.csv"


def default_runtime_npz_path(ticker: str) -> Path:
    stem = (
        f"labeled_dataset_XNAS_ITCH_{ticker.upper()}_mbo_"
        f"{compact_date(DATA_START_DATE)}_{compact_date(DATA_END_DATE)}"
    )
    return DATA_ROOT / "datasets" / f"{stem}_dynamic_preprocessed.npz"


def load_time_grid(ticker: str) -> np.ndarray | None:
    path = Path(RUNTIME_NPZ_PATH_OVERRIDE) if RUNTIME_NPZ_PATH_OVERRIDE else default_runtime_npz_path(ticker)
    if not path.exists():
        return None
    with np.load(path, allow_pickle=False) as data:
        if "time_grid" not in data:
            return None
        return np.asarray(data["time_grid"], dtype=float)


def normalize_grid(df: pd.DataFrame, *, ticker: str | None = None, model_type: str | None = None) -> pd.DataFrame:
    out = df.rename(columns={k: v for k, v in COLUMN_ALIASES.items() if k in df.columns}).copy()
    missing = [col for col in REQUIRED_GRID_COLUMNS if col not in out.columns]
    if missing:
        raise ValueError(f"Missing required toxic-CIF grid columns: {missing}")
    for col in [
        "orders", "submitted", "skipped", "filled", "unfilled", "canceled",
        "metric_ok", "metric_failed", "fill_rate", "mean_is_bps", "median_is_bps",
        "time_weighted_mean_is_bps", "opportunity_cost_mean_bps", "toxic_cost_mean_bps",
        "max_toxic_cif", "horizon_index", "cache_size", "cache_hits", "cache_misses",
    ]:
        if col in out.columns:
            out[col] = pd.to_numeric(out[col], errors="coerce")
    if "ticker" not in out.columns and ticker is not None:
        out["ticker"] = ticker.upper()
    if "model_type" not in out.columns and model_type is not None:
        out["model_type"] = model_type
    if "decision_logic" not in out.columns:
        out["decision_logic"] = "toxic_cif"
    if "orders" in out.columns:
        out["submitted_rate"] = out["submitted"] / out["orders"]
        out["skipped_rate"] = out["skipped"] / out["orders"]
        out["metric_failed_rate"] = out["metric_failed"] / out["orders"]
    if {"submitted_rate", "fill_rate"}.issubset(out.columns):
        out["real_fill_rate"] = out["submitted_rate"] * out["fill_rate"]
    return out


def add_horizon_seconds(df: pd.DataFrame, ticker: str) -> pd.DataFrame:
    out = df.copy()
    grid = load_time_grid(ticker)
    if grid is None:
        out["horizon_s"] = np.nan
        return out
    idx = out["horizon_index"].astype(int).clip(lower=0, upper=len(grid) - 1)
    out["horizon_s"] = idx.map(lambda i: float(grid[int(i)]))
    return out


def objective_column(df: pd.DataFrame, preferred: str) -> str:
    fallbacks = [preferred, "time_weighted_mean_is_bps", "mean_is_bps", "median_is_bps"]
    for col in fallbacks:
        if col in df.columns and df[col].notna().any():
            return col
    raise ValueError(f"None of the objective columns are available: {fallbacks}")


def feasible_mask(df: pd.DataFrame) -> pd.Series:
    mask = pd.Series(True, index=df.index)
    if MIN_REAL_FILL_RATE is not None and "real_fill_rate" in df.columns:
        mask &= df["real_fill_rate"] >= float(MIN_REAL_FILL_RATE)
    if MIN_SUBMITTED_RATE is not None and "submitted_rate" in df.columns:
        mask &= df["submitted_rate"] >= float(MIN_SUBMITTED_RATE)
    if MIN_FILL_RATE is not None and "fill_rate" in df.columns:
        mask &= df["fill_rate"] >= float(MIN_FILL_RATE)
    if MAX_METRIC_FAILED_RATE is not None and "metric_failed_rate" in df.columns:
        mask &= df["metric_failed_rate"] <= float(MAX_METRIC_FAILED_RATE)
    if MIN_SUBMITTED is not None and "submitted" in df.columns:
        mask &= df["submitted"] >= float(MIN_SUBMITTED)
    return mask.fillna(False)


def rank_grid(df: pd.DataFrame, objective_metric: str = OBJECTIVE_METRIC) -> pd.DataFrame:
    out = df.copy()
    obj_col = objective_column(out, objective_metric)
    out["objective_metric"] = obj_col
    out["objective"] = pd.to_numeric(out[obj_col], errors="coerce")
    out["feasible"] = feasible_mask(out) & out["objective"].notna()

    feasible_objectives = out.loc[out["feasible"], "objective"]
    if feasible_objectives.empty:
        best_objective = np.nan
        out["objective_delta_bps"] = np.nan
        out["near_best"] = False
    else:
        best_objective = float(feasible_objectives.min())
        out["objective_delta_bps"] = out["objective"] - best_objective
        out["near_best"] = (
            out["feasible"]
            & out["objective_delta_bps"].le(float(NEAR_BEST_TOL_BPS))
        )
    out["best_feasible_objective"] = best_objective
    out["near_best_tolerance_bps"] = float(NEAR_BEST_TOL_BPS)

    sort_cols = [col for col in TIEBREAK_ASCENDING if col in out.columns]
    ascending = [TIEBREAK_ASCENDING[col] for col in sort_cols]
    ranked = out.sort_values(
        ["near_best", "feasible", *sort_cols],
        ascending=[False, False, *ascending],
    ).reset_index(drop=True)
    ranked["rank_all"] = np.arange(1, len(ranked) + 1)

    feasible_idx = ranked[ranked["feasible"]].index
    ranked["rank_feasible"] = np.nan
    ranked.loc[feasible_idx, "rank_feasible"] = np.arange(1, len(feasible_idx) + 1)

    selected_idx = ranked[ranked["near_best"]].index
    ranked["rank_near_best"] = np.nan
    ranked.loc[selected_idx, "rank_near_best"] = np.arange(1, len(selected_idx) + 1)
    return ranked

def load_one_grid(ticker: str = TICKER, model_type: str = MODEL_TYPE) -> pd.DataFrame:
    path = Path(RESULT_PATH_OVERRIDE) if RESULT_PATH_OVERRIDE else default_result_path(ticker, model_type)
    if not path.exists():
        raise FileNotFoundError(path)
    df = pd.read_csv(path)
    df = normalize_grid(df, ticker=ticker, model_type=model_type)
    df = add_horizon_seconds(df, ticker)
    print(f"Loaded {len(df):,} rows from {path}")
    return df


## Load One Grid

Use this section to inspect one ticker/model pair and choose a setting.

In [ ]:
grid = load_one_grid(TICKER, MODEL_TYPE)
ranked = rank_grid(grid, OBJECTIVE_METRIC)

summary_cols = [
    "ticker", "model_type", "decision_logic", "max_toxic_cif", "horizon_index", "horizon_s",
    "objective", "objective_metric", "mean_is_bps", "median_is_bps", "time_weighted_mean_is_bps",
    "toxic_cost_mean_bps", "opportunity_cost_mean_bps", "submitted", "submitted_rate", "fill_rate",
    "metric_failed", "metric_failed_rate", "cache_size", "cache_hits", "cache_misses",
]
summary_cols = [c for c in summary_cols if c in ranked.columns]
ranked[summary_cols].head(15)


## Best Configuration by Horizon

This reduces the 2D grid to the best toxic-CIF threshold for each horizon.

In [ ]:
best_by_horizon = (
    ranked[ranked["feasible"]]
    .sort_values(["horizon_index", "objective"], ascending=[True, True])
    .groupby("horizon_index", as_index=False)
    .first()
    .sort_values("horizon_index")
)

cols = [
    "horizon_index", "horizon_s", "max_toxic_cif", "objective", "mean_is_bps",
    "median_is_bps", "time_weighted_mean_is_bps", "toxic_cost_mean_bps",
    "submitted", "submitted_rate", "fill_rate", "metric_failed_rate",
]
cols = [c for c in cols if c in best_by_horizon.columns]
best_by_horizon[cols]


## 2D Grid Heatmaps

Rows are horizons and columns are `max_toxic_cif`. Lower objective values are better. The second heatmap shows submitted rate to reveal whether a better objective comes from a meaningful trading region or just extreme filtering.

In [ ]:
def pivot_grid(frame: pd.DataFrame, value_col: str) -> pd.DataFrame:
    return (
        frame.pivot_table(
            index="horizon_index",
            columns="max_toxic_cif",
            values=value_col,
            aggfunc="mean",
        )
        .sort_index(axis=0)
        .sort_index(axis=1)
    )

obj_col = objective_column(ranked, OBJECTIVE_METRIC)
fig, axes = plt.subplots(1, 2, figsize=(15, 5), constrained_layout=True)

sns.heatmap(
    pivot_grid(ranked, obj_col),
    annot=True,
    fmt=".3f",
    cmap="viridis_r",
    ax=axes[0],
)
axes[0].set_title(f"{TICKER} {MODEL_TYPE}: {obj_col}")
axes[0].set_xlabel("max_toxic_cif")
axes[0].set_ylabel("horizon_index")

if "submitted_rate" in ranked.columns:
    sns.heatmap(
        pivot_grid(ranked, "submitted_rate"),
        annot=True,
        fmt=".1%",
        cmap="Blues",
        ax=axes[1],
    )
    axes[1].set_title("Submitted Rate")
    axes[1].set_xlabel("max_toxic_cif")
    axes[1].set_ylabel("horizon_index")
else:
    axes[1].axis("off")

plt.show()


## Threshold Curves

These line plots make it easier to see monotonicity and participation trade-offs across toxic-CIF thresholds.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5), constrained_layout=True)

sns.lineplot(
    data=ranked,
    x="max_toxic_cif",
    y="objective",
    hue="horizon_index",
    marker="o",
    palette="tab10",
    ax=axes[0],
)
axes[0].set_title(f"Objective: {objective_column(ranked, OBJECTIVE_METRIC)}")
axes[0].set_xlabel("max_toxic_cif")
axes[0].set_ylabel("lower is better")

if "submitted_rate" in ranked.columns:
    sns.lineplot(
        data=ranked,
        x="max_toxic_cif",
        y="submitted_rate",
        hue="horizon_index",
        marker="o",
        palette="tab10",
        legend=False,
        ax=axes[1],
    )
    axes[1].set_title("Submitted Rate")
    axes[1].set_xlabel("max_toxic_cif")
    axes[1].set_ylabel("submitted / orders")
else:
    axes[1].axis("off")

plt.show()


## Near-Optimal Region

A stable setting is often preferable to a one-cell winner. This cell lists configurations within a chosen tolerance of the best feasible objective.

In [ ]:
feasible = ranked[ranked["feasible"]].copy()
if feasible.empty:
    raise ValueError("No feasible rows. Relax the feasibility filters above.")

best_objective = float(feasible["objective"].iloc[0])
near = feasible[feasible["objective"] <= best_objective + float(NEAR_BEST_TOL_BPS)].copy()
near_cols = [
    "rank_near_best", "rank_feasible", "max_toxic_cif", "horizon_index", "horizon_s", "objective",
    "mean_is_bps", "median_is_bps", "time_weighted_mean_is_bps", "toxic_cost_mean_bps",
    "submitted", "submitted_rate", "fill_rate", "metric_failed_rate",
]
near_cols = [c for c in near_cols if c in near.columns]
near[near_cols]


## Selected Setting

This is the setting to use with `DeepHitToxicCIFDecisionLogic`.

In [ ]:
selected = feasible.iloc[0]
selected_dict = {
    "ticker": selected.get("ticker", TICKER),
    "model_type": selected.get("model_type", MODEL_TYPE),
    "decision_logic": "toxic_cif",
    "max_toxic_cif": float(selected["max_toxic_cif"]),
    "horizon_index": int(selected["horizon_index"]),
    "objective_metric": selected["objective_metric"],
    "objective": float(selected["objective"]),
}
selected_dict


In [ ]:
print("Use this in DeepHitToxicCIFDecisionLogic:")
print(
    "DeepHitToxicCIFDecisionLogic("
    f"max_toxic_cif={selected_dict['max_toxic_cif']:.6g}, "
    f"horizon_index={selected_dict['horizon_index']}"
    ")"
)


## Select Settings Across All Available Results

This scans the output tree, ranks each ticker/model file with the same objective and feasibility filters, and collects one selected row per file.

In [ ]:
def discover_result_files(output_root: Path = OUTPUT_ROOT) -> list[Path]:
    return sorted(Path(output_root).glob("*/*/deephit_toxic_cif_grid_search.csv"))


def load_and_select(path: Path) -> dict:
    ticker = path.parents[1].name
    model_type = path.parent.name
    df = pd.read_csv(path)
    df = normalize_grid(df, ticker=ticker, model_type=model_type)
    df = add_horizon_seconds(df, ticker)

    #df = df.loc[df['horizon_index'] == 29]
    df = df.loc[df['max_toxic_cif'] != 1]

    ranked_df = rank_grid(df, OBJECTIVE_METRIC)
    near_best_df = ranked_df[ranked_df["near_best"]]
    feasible_df = ranked_df[ranked_df["feasible"]]
    if not near_best_df.empty:
        chosen = near_best_df.iloc[0].copy()
        chosen["selection_warning"] = ""
    elif not feasible_df.empty:
        chosen = feasible_df.iloc[0].copy()
        chosen["selection_warning"] = "no_near_best_rows"
    else:
        chosen = ranked_df.iloc[0].copy()
        chosen["selection_warning"] = "no_feasible_rows"
    chosen["min_real_fill_rate_threshold"] = MIN_REAL_FILL_RATE
    chosen["max_metric_failed_rate_threshold"] = MAX_METRIC_FAILED_RATE
    chosen["near_best_tolerance_bps"] = float(NEAR_BEST_TOL_BPS)
    out = chosen.to_dict()
    out["result_path"] = str(path)
    return out

paths = discover_result_files(OUTPUT_ROOT)
print(f"Found {len(paths):,} toxic-CIF grid file(s).")

selected_rows = [load_and_select(path) for path in paths]
selected_all = pd.DataFrame(selected_rows)

selected_cols = [
    "ticker", "model_type", "decision_logic", "max_toxic_cif", "horizon_index", "horizon_s",
    "objective_metric", "objective", "best_feasible_objective", "objective_delta_bps",
    "near_best", "near_best_tolerance_bps", "mean_is_bps", "median_is_bps",
    "time_weighted_mean_is_bps", "toxic_cost_mean_bps", "opportunity_cost_mean_bps",
    "submitted", "submitted_rate", "fill_rate", "real_fill_rate",
    "metric_failed", "metric_failed_rate", "min_real_fill_rate_threshold",
    "max_metric_failed_rate_threshold",
    "selection_warning", "result_path",
]
selected_cols = [c for c in selected_cols if c in selected_all.columns]
selected_all = selected_all.sort_values(["ticker", "model_type"]).reset_index(drop=True)
selected_all[selected_cols]


## Export Selected Settings

The exported CSV is compatible with final-evaluation scripts after they are pointed to the toxic-CIF decision logic columns.

In [ ]:
EXPORT_SELECTED_PATH.parent.mkdir(parents=True, exist_ok=True)
selected_all[selected_cols].to_csv(EXPORT_SELECTED_PATH, index=False)
print(f"Wrote {len(selected_all):,} selected setting row(s) to {EXPORT_SELECTED_PATH}")


## Improvement vs Real Fill-Rate Constraint

Treat `max_toxic_cif = 1.0` as the baseline strategy. For each ticker/model and each minimum real fill-rate threshold, this cell selects the best feasible toxic-CIF setting and plots its objective improvement versus the baseline. Real fill rate is `submitted_rate * fill_rate`; positive improvement means the selected setting has lower implementation shortfall than baseline.


In [ ]:
# ---- Improvement over max_toxic_cif=1.0 baseline as real fill-rate threshold tightens ----
MODEL_TYPE_ORDER = ["gru", "gru_transformer", "transformer", "mamba"]
FILL_RATE_THRESHOLDS = np.round(np.arange(0.10, 0.95 + 1e-9, 0.05), 2)
BASELINE_MAX_TOXIC_CIF = 1.0
PREFERRED_BASELINE_HORIZON = 4

# Plot-specific feasibility filters. The curve is primarily meant to vary
# minimum realized fill rate, so these default to relaxed values instead of
# inheriting the global selection filters above.
IMPROVEMENT_MIN_SUBMITTED_RATE = None
IMPROVEMENT_MIN_FILL_RATE = None
IMPROVEMENT_MAX_METRIC_FAILED_RATE = None
IMPROVEMENT_MIN_SUBMITTED = None

# Keep this plot objective-first. The global TIEBREAK_ASCENDING is intentionally
# stability-oriented for final setting export, but this curve is meant to show
# the best achievable improvement at each realized fill-rate threshold.
IMPROVEMENT_TIEBREAK_ASCENDING = {
    "objective": True,
    "metric_failed_rate": True,
    "submitted": False,
    "toxic_cost_mean_bps": True,
    "max_toxic_cif": True,
    "horizon_index": True,
}


def _baseline_row(df: pd.DataFrame) -> pd.Series | None:
    baseline = df[np.isclose(df["max_toxic_cif"].astype(float), BASELINE_MAX_TOXIC_CIF)]
    if baseline.empty:
        return None
    preferred = baseline[baseline["horizon_index"].astype(int).eq(int(PREFERRED_BASELINE_HORIZON))]
    if not preferred.empty:
        return preferred.sort_values("horizon_index").iloc[0]
    return baseline.sort_values("horizon_index").iloc[0]


def _improvement_feasible_mask(df: pd.DataFrame, min_real_fill_rate: float) -> pd.Series:
    mask = df["real_fill_rate"].ge(float(min_real_fill_rate))
    if IMPROVEMENT_MIN_SUBMITTED_RATE is not None and "submitted_rate" in df.columns:
        mask &= df["submitted_rate"].ge(float(IMPROVEMENT_MIN_SUBMITTED_RATE))
    if IMPROVEMENT_MIN_FILL_RATE is not None and "fill_rate" in df.columns:
        mask &= df["fill_rate"].ge(float(IMPROVEMENT_MIN_FILL_RATE))
    if IMPROVEMENT_MAX_METRIC_FAILED_RATE is not None and "metric_failed_rate" in df.columns:
        mask &= df["metric_failed_rate"].le(float(IMPROVEMENT_MAX_METRIC_FAILED_RATE))
    if IMPROVEMENT_MIN_SUBMITTED is not None and "submitted" in df.columns:
        mask &= df["submitted"].ge(float(IMPROVEMENT_MIN_SUBMITTED))
    return mask.fillna(False)


def _rank_for_threshold(df: pd.DataFrame, min_real_fill_rate: float) -> pd.DataFrame:
    ranked_df = df.copy()

    #ranked_df = ranked_df.loc[ranked_df['horizon_index'] == 24]

    obj_col = objective_column(ranked_df, OBJECTIVE_METRIC)
    ranked_df["objective_metric"] = obj_col
    ranked_df["objective"] = pd.to_numeric(ranked_df[obj_col], errors="coerce")
    ranked_df["real_fill_rate"] = ranked_df["submitted_rate"] * ranked_df["fill_rate"]
    ranked_df["feasible"] = (
        _improvement_feasible_mask(ranked_df, min_real_fill_rate)
        & ranked_df["objective"].notna()
    )
    sort_cols = ["feasible"] + [
        col for col in IMPROVEMENT_TIEBREAK_ASCENDING if col in ranked_df.columns
    ]
    ascending = [False] + [
        IMPROVEMENT_TIEBREAK_ASCENDING[col] for col in sort_cols[1:]
    ]
    return ranked_df.sort_values(sort_cols, ascending=ascending).reset_index(drop=True)


def improvement_curve_for_file(path: Path) -> pd.DataFrame:
    ticker = path.parents[1].name
    model_type = path.parent.name
    df = pd.read_csv(path)
    df = normalize_grid(df, ticker=ticker, model_type=model_type)
    df = add_horizon_seconds(df, ticker)
    obj_col = objective_column(df, OBJECTIVE_METRIC)
    df["real_fill_rate"] = df["submitted_rate"] * df["fill_rate"]

    base = _baseline_row(df)
    if base is None or pd.isna(base.get(obj_col)):
        return pd.DataFrame()
    baseline_objective = float(base[obj_col])
    baseline_horizon = int(base["horizon_index"])

    rows = []
    for threshold in FILL_RATE_THRESHOLDS:
        ranked_at_threshold = _rank_for_threshold(df, float(threshold))
        feasible = ranked_at_threshold[ranked_at_threshold["feasible"]]
        if feasible.empty:
            rows.append({
                "ticker": ticker,
                "model_type": model_type,
                "real_fill_rate_threshold": float(threshold),
                "baseline_objective": baseline_objective,
                "baseline_horizon_index": baseline_horizon,
                "selected_objective": np.nan,
                "improvement_bps": np.nan,
                "selected_max_toxic_cif": np.nan,
                "selected_horizon_index": np.nan,
                "selected_real_fill_rate": np.nan,
            })
            continue
        selected = feasible.iloc[0]
        selected_objective = float(selected["objective"])
        rows.append({
            "ticker": ticker,
            "model_type": model_type,
            "real_fill_rate_threshold": float(threshold),
            "baseline_objective": baseline_objective,
            "baseline_horizon_index": baseline_horizon,
            "selected_objective": selected_objective,
            "improvement_bps": baseline_objective - selected_objective,
            "selected_max_toxic_cif": float(selected["max_toxic_cif"]),
            "selected_horizon_index": int(selected["horizon_index"]),
            "selected_real_fill_rate": float(selected["real_fill_rate"]),
        })
    return pd.DataFrame(rows)

improvement_paths = discover_result_files(OUTPUT_ROOT)
improvement_curves = pd.concat(
    [curve for path in improvement_paths if not (curve := improvement_curve_for_file(path)).empty],
    ignore_index=True,
) if improvement_paths else pd.DataFrame()

if improvement_curves.empty:
    raise ValueError(f"No improvement curves could be built under {OUTPUT_ROOT}")

improvement_curves.head()


In [ ]:
model_types = [m for m in MODEL_TYPE_ORDER if m in set(improvement_curves["model_type"])]
extra_models = sorted(set(improvement_curves["model_type"]) - set(model_types))
model_types.extend(extra_models)

MODEL_LABELS = {
    "gru": "GRU",
    "gru_transformer": "GRU + Transformer",
    "transformer": "Transformer",
    "mamba": "Mamba",
}
OBJECTIVE_LABELS = {
    "mean_is_bps": "Mean IS",
    "median_is_bps": "Median IS",
    "time_weighted_mean_is_bps": "Time-weighted mean IS",
    "toxic_cost_mean_bps": "Toxic-cost IS",
}
metric_label = OBJECTIVE_LABELS.get(OBJECTIVE_METRIC, OBJECTIVE_METRIC)

tickers = sorted(improvement_curves["ticker"].dropna().unique())
palette = sns.color_palette("tab20", n_colors=max(len(tickers), 1))
color_for_ticker = dict(zip(tickers, palette))

finite_y = improvement_curves["improvement_bps"].replace([np.inf, -np.inf], np.nan).dropna()
if finite_y.empty:
    y_limits = None
else:
    y_abs = float(np.nanmax(np.abs(finite_y)))
    # y_limits = (-1.12 * y_abs, 1.12 * y_abs)
    y_limits = (0.0001, 1.12 * y_abs)

fig, axes = plt.subplots(2, 2, figsize=(15, 10), sharex=True, sharey=True)
axes = axes.ravel()

for ax, model_type in zip(axes, model_types):
    frame = improvement_curves[improvement_curves["model_type"].eq(model_type)]
    available_tickers = set(frame["ticker"].dropna())
    for ticker, group in frame.groupby("ticker"):
        group = group.sort_values("real_fill_rate_threshold")
        ax.plot(
            group["real_fill_rate_threshold"],
            group["improvement_bps"],
            marker="o",
            markersize=3.2,
            linewidth=1.25,
            alpha=0.78,
            color=color_for_ticker.get(ticker),
            label=ticker,
        )
    ax.axhline(0.0, color="0.25", linewidth=0.9, linestyle="--", alpha=0.8)
    ax.set_yscale("log")#"symlog", linthresh=0.05, linscale=0.65)
    ax.set_title(MODEL_LABELS.get(model_type, model_type), fontsize=12, pad=6)
    ax.grid(True, alpha=0.22, which="both")
    ax.xaxis.set_major_formatter(plt.FuncFormatter(lambda x, _: f"{x:.0%}"))
    ax.tick_params(axis="both", labelsize=9)
    if y_limits is not None:
        ax.set_ylim(*y_limits)
    missing_tickers = sorted(set(tickers) - available_tickers)
    if missing_tickers:
        ax.text(
            0.99,
            0.03,
            "No baseline: " + ", ".join(missing_tickers),
            transform=ax.transAxes,
            ha="right",
            va="bottom",
            fontsize=7.5,
            color="0.45",
        )

for ax in axes[len(model_types):]:
    ax.axis("off")

fig.supxlabel("Minimum realized fill rate", fontsize=12, y=0.072)
fig.supylabel(f"Improvement vs baseline ({metric_label}, bps)", fontsize=12, x=0.028)
fig.suptitle(
    "Best Toxic-CIF Strategy Improvement Relative to max_toxic_cif = 1 Baseline",
    fontsize=14,
    y=0.990,
)

from matplotlib.lines import Line2D
legend_handles = [
    Line2D([0], [0], color=color_for_ticker[ticker], marker="o", linewidth=1.25, markersize=3.5, label=ticker)
    for ticker in tickers
]
if legend_handles:
    fig.legend(
        handles=legend_handles,
        labels=tickers,
        loc="upper center",
        bbox_to_anchor=(0.5, 0.948),
        ncol=min(6, len(tickers)),
        frameon=False,
        fontsize=9,
        handlelength=2.0,
        columnspacing=1.4,
    )

fig.tight_layout(rect=[0.02, 0.06, 1.0, 0.92], h_pad=1.55, w_pad=1.35)
plt.show()